# 🏭 Warehouse Digital Twin - Complete Demo

This notebook demonstrates the full capabilities of the Warehouse Digital Twin system:
- ✅ Create warehouses with 100,000+ aisles
- ✅ ML-powered optimization
- ✅ 3D visualization
- ✅ Geospatial mapping
- ✅ AGV pathfinding

## 📦 Setup

Install dependencies (for Colab)

In [ ]:
# Uncomment for Google Colab
# !pip install plotly numpy pandas matplotlib scikit-learn pydeck geopandas shapely ipywidgets networkx pydantic --quiet

import warnings
warnings.filterwarnings('ignore')

print("✅ Setup complete!")

## 🚀 Initialize System

In [ ]:
from main import WarehouseDigitalTwin, generate_sample_data
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Create Digital Twin instance
twin = WarehouseDigitalTwin()

# Global variable to store products
products_df = None

## 🏗️ Create Warehouse

Create a warehouse with **10,000 aisles** (scalable to 100,000+!)

In [ ]:
# Interactive Warehouse Creation

# Create widgets
warehouse_name = widgets.Text(
    value='Mega Distribution Center',
    description='Name:',
    style={'description_width': '120px'}
)

warehouse_length = widgets.FloatText(
    value=200.0,
    description='Length (m):',
    style={'description_width': '120px'}
)

warehouse_width = widgets.FloatText(
    value=150.0,
    description='Width (m):',
    style={'description_width': '120px'}
)

warehouse_height = widgets.FloatText(
    value=12.0,
    description='Height (m):',
    style={'description_width': '120px'}
)

num_aisles = widgets.IntText(
    value=10000,
    description='Num Aisles:',
    style={'description_width': '120px'}
)

create_button = widgets.Button(
    description='Create Warehouse',
    button_style='success',
    icon='building'
)

warehouse_output = widgets.Output()

def on_create_warehouse(b):
    with warehouse_output:
        clear_output()
        try:
            warehouse = twin.create_warehouse(
                name=warehouse_name.value,
                length=warehouse_length.value,
                width=warehouse_width.value,
                height=warehouse_height.value,
                num_aisles=num_aisles.value,
                aisle_width=3.0,
                rack_height=10.0
            )
            
            print(f"\n📊 Warehouse Statistics:")
            print(f"   Total volume: {warehouse.dimensions.volume:,.0f} m³")
            print(f"   Estimated shelves: {warehouse.estimated_total_shelves:,}")
            print(f"   Storage systems: {len(warehouse.storage_systems)}")
        except Exception as e:
            print(f"❌ Error creating warehouse: {e}")

create_button.on_click(on_create_warehouse)

# Display widgets
display(widgets.VBox([
    warehouse_name,
    warehouse_length,
    warehouse_width,
    warehouse_height,
    num_aisles,
    create_button,
    warehouse_output
]))

## 📦 Generate Sample Products

In [ ]:
# Interactive Product Loading

# Create widgets
csv_path_widget = widgets.Text(
    value='test_data_unseen_expanded.csv',
    description='CSV Path:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

load_csv_button = widgets.Button(
    description='Load from CSV',
    button_style='primary',
    icon='upload'
)

generate_button = widgets.Button(
    description='Generate Sample Data',
    button_style='info',
    icon='magic'
)

product_output = widgets.Output()

def on_load_csv(b):
    global products_df
    with product_output:
        clear_output()
        try:
            csv_path = csv_path_widget.value
            
            # Check if file exists
            if not Path(csv_path).exists():
                print(f"❌ Error: File not found: {csv_path}")
                print(f"\nPlease check the file path and try again.")
                print(f"Or use 'Generate Sample Data' button to create test data.")
                return
            
            # Load products from CSV
            products = twin.load_products_from_csv(csv_path)
            products_df = twin.products_df
            
            print(f"✅ Successfully loaded {len(products_df)} products from CSV")
            display(products_df.head(10))
            
        except FileNotFoundError as e:
            print(f"❌ File Error: {e}")
            print(f"\nPlease check the file path and try again.")
        except Exception as e:
            print(f"❌ Error loading CSV: {e}")
            print(f"\nPlease check the CSV format and try again.")

def on_generate_sample(b):
    global products_df
    with product_output:
        clear_output()
        try:
            # Generate sample data
            products_df = generate_sample_data(5000)
            
            print(f"✅ Generated {len(products_df)} sample products")
            display(products_df.head(10))
            
        except Exception as e:
            print(f"❌ Error generating sample data: {e}")

load_csv_button.on_click(on_load_csv)
generate_button.on_click(on_generate_sample)

# Display widgets
display(widgets.VBox([
    csv_path_widget,
    widgets.HBox([load_csv_button, generate_button]),
    product_output
]))

## 🎓 Train Optimization Model

In [ ]:
# Train ML model
if products_df is None:
    print("❌ Error: No products loaded. Please load products or generate sample data first.")
else:
    twin.train_optimizer(products_df)

## ⚙️ Optimize Product Allocation

In [ ]:
# Optimize allocation
if products_df is None:
    print("❌ Error: No products loaded. Please load products or generate sample data first.")
else:
    results = twin.optimize_allocation(products_df)
    
    # Display results
    print("\n📊 Allocation Results:")
    results.head(10)

## 📊 Utilization Analysis

In [ ]:
# Create utilization chart
util_fig = twin.visualize_utilization()
util_fig.show()

## 🎨 3D Warehouse Visualization

In [ ]:
# Create 3D visualization
fig_3d = twin.visualize_warehouse_3d(
    show_products=True,
    max_products=1000  # Limit for performance
)

fig_3d.show()

## 🔍 Product Clustering

In [ ]:
# Run clustering
clustered_df = twin.run_clustering(n_clusters=5)

# Visualize clusters
cluster_fig = twin.visualize_clusters()
cluster_fig.show()

## 🗺️ Geospatial 3D Map

In [ ]:
# Create 3D geospatial map
deck = twin.create_geospatial_map(style="3d")

# Display map
deck.show()

## 📈 Performance Dashboard

In [ ]:
# Create comprehensive dashboard
dashboard = twin.create_dashboard()
dashboard.show()

## 🚗 AGV Pathfinding Demo

In [ ]:
from core import PathfindingGrid, generate_warehouse_navigation_grid

# Generate navigation grid
print("Generating navigation grid...")
nodes, segments = generate_warehouse_navigation_grid(
    warehouse_length=200,
    warehouse_width=150,
    num_aisles=20,  # Use smaller number for demo
    aisle_width=3.0
)

print(f"✅ Generated {len(nodes)} nodes and {len(segments)} segments")

# Create pathfinding system
pathfinder = PathfindingGrid(nodes, segments)

# Find path between two points
start_node = "node_a0_n0"
goal_node = "node_a10_n20"

path = pathfinder.find_path(start_node, goal_node)

if path:
    print(f"\n🎯 Path found!")
    print(f"   Nodes: {len(path.nodes)}")
    print(f"   Distance: {path.total_distance:.1f}m")
    print(f"   Estimated time: {path.estimated_time:.1f}s")
else:
    print("❌ No path found")

## 🎉 Summary

This demo showcased:
- ✅ Warehouse creation with **10,000 aisles**
- ✅ ML-powered optimization
- ✅ 3D warehouse visualization
- ✅ Product clustering analysis
- ✅ Geospatial 3D mapping
- ✅ AGV pathfinding simulation

### Next Steps:
- Scale to 100,000+ aisles
- Add real product data
- Implement FastAPI endpoints
- Add real-time AGV simulation